# MultiMNIST — complete comparison and ablation table

Use **Runtime → Change runtime type → A100 GPU**. All notebook text and logs are in English.

**Continuing the existing experiment:** run cells **1–5**, skip **6** if Ours is already complete, then run **7–15** in order. Cell 5 loads the saved validation selection by default; it does not retune. The default experiment name, dataset archive and training-code revision are unchanged, so your existing three-seed Ours results remain usable.

**Starting from scratch:** enable `RUN_TUNING` in cell 1, then run cells **1–15**. This also performs the existing validation search and final Ours training.

Cells 8–10 run all six comparison baselines. Cells 11–14 run all 13 ablation configurations of **Entropic LMO-MGDA**. Every configuration uses seeds 42, 43 and 44, 100 epochs, and all 10,000 training images; test results use the separate 1,000-image test split. Ablations change one setting at a time around the saved Ours configuration, without retuning each variant.

Each training process streams **one metric log per epoch, with no batch progress bars**. Completed matching runs are skipped; incomplete runs resume from their own checkpoints. Only run one notebook against this output directory at a time.

**Budget:** six baselines × three seeds = 18 runs; 13 ablations × three seeds = 39 runs. With Ours already complete, that is at most **57 additional 100-epoch runs**. Cell 7 shows remaining work. Baseline-equivalent ablation settings are kept as separate, explicitly tagged control runs.

The code reference is pinned to MOON commit `37319d1`. The paper's beta grid is used for our eta ablation. FAMO uses the released code defaults because an independently selected MultiMNIST FAMO configuration is not specified in the paper. Sources and exact settings appear before cell 7. Exact reproduction of published accuracy is not guaranteed because the paper's generated dataset was not released.


In [3]:
#@title 1. Experiment settings
REPO_URL = "https://github.com/alirezamirrokni/LMO-MOO.git"
REPO_COMMIT = "f6e16f360b792db58763f994013428e6a1a4b5c2"
EXPERIMENT_NAME = "multimnist_a100_v2" #@param {type:"string"}
# Enable only for a new experiment without tuning/selection.json.
RUN_TUNING = False #@param {type:"boolean"}
SEEDS = [42, 43, 44]
EPOCHS = 100
BATCH_SIZE = 256
# Keep the existing v1 dataset; these are input files, not old model checkpoints.
DATA_CACHE_EXPERIMENT = "multimnist_a100_v1"


In [4]:
#@title 2. Check A100, mount Drive and enable live console logs
import os, sys, json, subprocess, shutil, time, hashlib, zipfile, csv
from pathlib import Path
import torch
from google.colab import drive

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > A100 GPU."
GPU_NAME = torch.cuda.get_device_name(0)
assert "A100" in GPU_NAME, f"Current GPU: {GPU_NAME}. Select A100 and reconnect."
print("GPU:", GPU_NAME, "| PyTorch:", torch.__version__)
drive.mount("/content/drive")
assert EXPERIMENT_NAME and Path(EXPERIMENT_NAME).name == EXPERIMENT_NAME and EXPERIMENT_NAME not in {".", ".."}
DRIVE_ROOT = Path("/content/drive/MyDrive/LMO-MOO")
RUN_ROOT = DRIVE_ROOT / EXPERIMENT_NAME
OUTPUT_ROOT = RUN_ROOT / "results" / "multimnist"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPO = Path("/content/LMO-MOO-v2")
DATA = Path("/content/LMO-MOO-data/multimnist")
print("Persistent outputs:", OUTPUT_ROOT)


import codecs, signal, shlex, uuid
LOG_DIR = RUN_ROOT / "logs"
LOG_DIR.mkdir(exist_ok=True)

def run_live(cmd, *, cwd=None, label="process"):
    """Forward stdout/stderr chunks immediately, preserving carriage returns.

    Raw console logs are also saved to Drive. An interrupted cell stops the
    entire subprocess group, including children launched by the suite.
    """
    cmd = list(map(str, cmd))
    print("$ " + shlex.join(cmd), flush=True)
    safe_label = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in label)
    log_path = LOG_DIR / (time.strftime("%Y%m%d_%H%M%S") + "_" + safe_label + "_" + uuid.uuid4().hex[:6] + ".log")
    env = os.environ.copy()
    env.update(PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8")
    print("Console log:", log_path, flush=True)
    with log_path.open("wb", buffering=0) as log:
        process = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, bufsize=0,
                                   start_new_session=True)
        decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
        try:
            while True:
                chunk = os.read(process.stdout.fileno(), 8192)
                if not chunk:
                    break
                # Display first so a slow Drive write cannot hide current output.
                sys.stdout.write(decoder.decode(chunk))
                sys.stdout.flush()
                log.write(chunk)
            sys.stdout.write(decoder.decode(b"", final=True))
            sys.stdout.flush()
            returncode = process.wait()
        except BaseException:
            if process.poll() is None:
                try:
                    os.killpg(process.pid, signal.SIGTERM)
                except ProcessLookupError:
                    pass
                try:
                    process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    try:
                        os.killpg(process.pid, signal.SIGKILL)
                    except ProcessLookupError:
                        pass
                    process.wait()
            raise
        finally:
            process.stdout.close()
    print(f"\n[{label}] Exit code: {returncode}", flush=True)
    if returncode:
        raise subprocess.CalledProcessError(returncode, cmd)
    return log_path


GPU: NVIDIA A100-SXM4-80GB | PyTorch: 2.11.0+cu128
Mounted at /content/drive
Persistent outputs: /content/drive/MyDrive/LMO-MOO/multimnist_a100_v2/results/multimnist


In [5]:
#@title 3. Clone the corrected code and install dependencies
if not REPO.exists():
    run_live(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", "--sparse", "--progress", REPO_URL, REPO], label="clone")
else:
    assert (REPO / ".git").exists(), f"{REPO} is not a Git checkout."
    origin = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
    assert origin == REPO_URL
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True)
    assert not dirty, "Save local tracked-file edits before rerunning setup."
if subprocess.run(["git", "-C", str(REPO), "cat-file", "-e", REPO_COMMIT + "^{commit}"], capture_output=True).returncode:
    run_live(["git", "-C", REPO, "fetch", "origin", REPO_COMMIT], label="fetch")
run_live(["git", "-C", REPO, "sparse-checkout", "set", "experiments", "methods", "scripts"], label="checkout-files")
run_live(["git", "-C", REPO, "checkout", "--detach", REPO_COMMIT], label="checkout-revision")
# Preserve Colab's installed CUDA-compatible torch and torchvision pair.
run_live([sys.executable, "-m", "pip", "install", "-r", REPO / "requirements-modern.txt", "pandas"], label="dependencies")
run_live([sys.executable, "-u", "-c", "import torch, torchvision; from experiments.multimnist.trainer import parser; print('Imports OK:', torch.__version__, torchvision.__version__)"], cwd=REPO, label="import-check")
os.chdir(REPO)
def run_script(filename, *args):
    return run_live([sys.executable, "-u", REPO / filename, *args], cwd=REPO, label=Path(filename).stem)

def atomic_json(path, value):
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(value, indent=2) + "\n")
    tmp.replace(path)

identity = {"repo": REPO_URL, "commit": REPO_COMMIT, "epochs": EPOCHS,
            "batch_size": BATCH_SIZE, "seeds": SEEDS, "dataset_seed": 2026,
            "train_samples": 10000, "test_samples": 1000}
manifest = RUN_ROOT / "experiment.json"
if manifest.exists():
    assert json.loads(manifest.read_text()) == identity, "Experiment settings changed. Use a new EXPERIMENT_NAME."
else:
    atomic_json(manifest, identity)
versions = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(RUN_ROOT / ("environment_" + time.strftime("%Y%m%d_%H%M%S") + ".txt")).write_text(versions)
print("Pinned code revision:", REPO_COMMIT)


$ git clone --depth 1 --filter=blob:none --no-checkout --sparse --progress https://github.com/alirezamirrokni/LMO-MOO.git /content/LMO-MOO-v2
Console log: /content/drive/MyDrive/LMO-MOO/multimnist_a100_v2/logs/20260924_183811_clone_f18588.log
Cloning into '/content/LMO-MOO-v2'...
remote: Enumerating objects: 21, done.        
remote: Counting objects: 100% (21/21), done.        
remote: Compressing objects: 100% (20/20), done.        
remote: Total 21 (delta 0), reused 16 (delta 0), pack-reused 0 (from 0)        
Receiving objects: 100% (21/21), 6.08 KiB | 6.08 MiB/s, done.

[clone] Exit code: 0
$ git -C /content/LMO-MOO-v2 sparse-checkout set experiments methods scripts
Console log: /content/drive/MyDrive/LMO-MOO/multimnist_a100_v2/logs/20260924_183812_checkout-files_60389f.log

[checkout-files] Exit code: 0
$ git -C /content/LMO-MOO-v2 checkout --detach f6e16f360b792db58763f994013428e6a1a4b5c2
Console log: /content/drive/MyDrive/LMO-MOO/multimnist_a100_v2/logs/20260924_183812_checkou

In [6]:
#@title 4. Prepare and cache the fixed dataset
CACHE = DRIVE_ROOT / DATA_CACHE_EXPERIMENT / "data"
CACHE.mkdir(parents=True, exist_ok=True)
ARCHIVE = CACHE / "multimnist_seed2026.zip"
LOCAL_ARCHIVE = Path("/content/multimnist_seed2026.zip")
if not ARCHIVE.exists():
    # Remove only a partial generated dataset from an interrupted preparation.
    if DATA.exists():
        shutil.rmtree(DATA)
    run_script("prepare_multimnist.py", "--download", "--output", DATA,
               "--mnist-root", "/content/LMO-MOO-data/mnist",
               "--train-samples", 10000, "--test-samples", 1000, "--seed", 2026)
    with zipfile.ZipFile(LOCAL_ARCHIVE, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(DATA.rglob("*")):
            if p.is_file():
                z.write(p, p.relative_to(DATA))
    print("Saving the dataset archive to Drive...", flush=True)
    partial = ARCHIVE.with_suffix(".zip.partial")
    shutil.copy2(LOCAL_ARCHIVE, partial)
    partial.replace(ARCHIVE)
else:
    print("Restoring the dataset archive from Drive...", flush=True)
    shutil.copy2(ARCHIVE, LOCAL_ARCHIVE)

digest = hashlib.sha256(LOCAL_ARCHIVE.read_bytes()).hexdigest()
hash_file = CACHE / "dataset_archive.sha256"
if hash_file.exists():
    assert hash_file.read_text().strip() == digest, "Dataset archive checksum mismatch."
else:
    hash_file.write_text(digest + "\n")
# Reuse local data within a session; restore it after reconnecting.
marker = DATA.parent / "archive.sha256"
if not (DATA.exists() and marker.exists() and marker.read_text().strip() == digest):
    if DATA.exists():
        shutil.rmtree(DATA)
    DATA.mkdir(parents=True)
    with zipfile.ZipFile(LOCAL_ARCHIVE) as z:
        assert z.testzip() is None, "Damaged dataset archive."
        for info in z.infolist():
            target = (DATA / info.filename).resolve()
            assert target.is_relative_to(DATA.resolve()), "Unsafe archive entry."
        for info in z.infolist():
            z.extract(info, DATA)
    marker.write_text(digest + "\n")
for split, expected in [("train", 10000), ("test", 1000)]:
    with (DATA / split / "labels.csv").open() as f:
        rows = list(csv.reader(f))
    assert len(rows) == expected
    assert all((DATA / split / "2" / row[0]).is_file() for row in rows)
    print(split, len(rows), "images")
print("Dataset ready on local disk:", DATA)


Restoring the dataset archive from Drive...
train 10000 images
test 1000 images
Dataset ready on local disk: /content/LMO-MOO-data/multimnist


In [7]:
#@title 5. Load the saved Ours configuration (or run validation tuning)
TUNING_ROOT = RUN_ROOT / "tuning"
SELECTION_FILE = TUNING_ROOT / "selection.json"
COMMON = ["--data-path", DATA, "--device", "cuda:0", "--epochs", EPOCHS,
          "--batch-size", BATCH_SIZE, "--workers", 0, "--threads", 4, "--cache-data"]
if RUN_TUNING:
    run_script("run_multimnist_tune.py", "--output-root", TUNING_ROOT, *COMMON)
assert SELECTION_FILE.exists(), "No selected configuration yet. Enable RUN_TUNING and run this cell."
selection = json.loads(SELECTION_FILE.read_text())
assert selection["test_used"] is False
assert selection["settings"]["selection"] == "validation"
SELECTED = selection["selected"]
assert SELECTED["epochs"] == EPOCHS and SELECTED["batch_size"] == BATCH_SIZE
# Verify that selection belongs to the currently restored training images.
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from experiments.multimnist.trainer import data_fingerprint
assert selection["settings"]["train_sha256"] == data_fingerprint(DATA, splits=("train",))
print("Selected configuration:", json.dumps(SELECTED, indent=2))
print("Final validation metrics (not test results):", selection["validation"])
# Prevent accidental mixing of existing final runs with a new selection.
final_manifest = RUN_ROOT / "final_configuration.json"
if final_manifest.exists():
    assert json.loads(final_manifest.read_text()) == SELECTED, "Selection changed. Use a new experiment directory."
else:
    atomic_json(final_manifest, SELECTED)
OURS_FLAGS = ["--lr", SELECTED["lr"], "--eta", SELECTED["eta"], "--alpha", SELECTED["alpha"],
              "--oracle", SELECTED["oracle"], "--weights", SELECTED["weights"], "--momentum", SELECTED["momentum"]]


$ /usr/bin/python3 -u /content/LMO-MOO-v2/run_multimnist_tune.py --output-root /content/drive/MyDrive/LMO-MOO/multimnist_a100_v2/tuning --data-path /content/LMO-MOO-data/multimnist --device cuda:0 --epochs 100 --batch-size 256 --workers 0 --threads 4 --cache-data
Console log: /content/drive/MyDrive/LMO-MOO/multimnist_a100_v2/logs/20260924_183832_run_multimnist_tune_2cf2d7.log

Screen 1/8: lr=0.001, alpha=0.1

/usr/bin/python3 -u /content/LMO-MOO-v2/run_multimnist.py --method ours --seed 42 --lr 0.001 --eta 0.0001 --alpha 0.1 --selection validation --metric sample --split-seed 2026 --val-fraction 0.1 --epochs 100 --stop-after-epoch 40 --batch-size 256 --data-path /content/LMO-MOO-data/multimnist --output-root /content/drive/MyDrive/LMO-MOO/multimnist_a100_v2/tuning --tag lr-0.001-alpha-0.1 --resume --device cuda:0 --workers 0 --threads 4 --cache-data
Run: method=ours, seed=42, selection=validation, lr=0.001, eta=0.0001, alpha=0.1
[MultiMNISTDataset] split=train, num_samples=10000, img_d

The following cell starts fresh final models on **all 10,000 training images**, using the selected hyperparameters. It never loads a tuning checkpoint. Subsequent reruns resume only these final runs. At the end of each epoch, the log reports train/test accuracies, training losses, task weights, inner gap and elapsed seconds.


In [8]:
#@title 6. Train Entropic LMO-MGDA on all training data with three seeds
started = time.monotonic()
run_script("run_multimnist_suite.py", "--suite", "main", "--methods", "ours",
           "--seeds", *SEEDS, "--output-root", OUTPUT_ROOT, *COMMON, *OURS_FLAGS)
print(f"This cell took {(time.monotonic() - started) / 60:.1f} minutes.")
print("Saved results:", OUTPUT_ROOT)


$ /usr/bin/python3 -u /content/LMO-MOO-v2/run_multimnist_suite.py --suite main --methods ours --seeds 42 43 44 --output-root /content/drive/MyDrive/LMO-MOO/multimnist_a100_v2/results/multimnist --data-path /content/LMO-MOO-data/multimnist --device cuda:0 --epochs 100 --batch-size 256 --workers 0 --threads 4 --cache-data --lr 0.01 --eta 0.0001 --alpha 0.1 --oracle spectral --weights entropic --momentum blended
Console log: /content/drive/MyDrive/LMO-MOO/multimnist_a100_v2/logs/20260924_185033_run_multimnist_suite_49dd39.log

Run 1/3: ours, seed 42, main
/usr/bin/python3 -u /content/LMO-MOO-v2/run_multimnist.py --method ours --seed 42 --tag main --data-path /content/LMO-MOO-data/multimnist --output-root /content/drive/MyDrive/LMO-MOO/multimnist_a100_v2/results/multimnist --resume --device cuda:0 --epochs 100 --batch-size 256 --workers 0 --threads 4 --cache-data --lr 0.01 --eta 0.0001 --alpha 0.1 --oracle spectral --weights entropic --momentum blended
Run: method=ours, seed=42, selection=

## Reference settings and what is being reproduced

| Method | Model optimizer | Model LR | Task-weight LR | Task-weight gamma |
| --- | --- | --- | --- | --- |
| MGDA | Adam | 0.001 | Not applicable | Not applicable |
| MGDA + Muon | Muon + auxiliary Adam | 0.001 | Not applicable | Not applicable |
| FAMO | Adam | 0.001 | 0.025 | 0.01 |
| FAMO + Muon | Muon + auxiliary Adam | 0.001 | 0.025 | 0.01 |
| Muon (LS) | Muon + auxiliary Adam | 0.001 | Fixed equal weights | Not applicable |
| MOON | Muon + auxiliary Adam | 0.001 | 0.0001 | 0.001 |

All comparison methods use the released ViT, 100 epochs, batch size 256, and `StepLR(step_size=100, gamma=0.5)`. The LR reduction occurs after the last training epoch. Muon uses momentum 0.95, Nesterov updates, five Newton–Schulz steps, and weight decay 0.001; auxiliary Adam uses betas (0.9, 0.95). Plain Adam uses its released trainer defaults. MGDA uses shared gradients and no gradient normalization. Our three seeds are 42, 43 and 44.

- **MOON settings:** [official launch script](https://github.com/KunlinLyu/MOON/blob/37319d14765595577b9b01fe91d6fcfca5ffbd7c/experiments/multimnist/run_muon_vit.sh) and [official trainer](https://github.com/KunlinLyu/MOON/blob/37319d14765595577b9b01fe91d6fcfca5ffbd7c/experiments/multimnist/trainer_muon_vit.py).
- **FAMO defaults:** [official common parser](https://github.com/KunlinLyu/MOON/blob/37319d14765595577b9b01fe91d6fcfca5ffbd7c/experiments/utils.py). These are code defaults, not a claim that we recovered the authors' selected FAMO run configuration.
- **Muon combinations:** Appendix F.2, Table 5 of the [MOON paper](https://arxiv.org/pdf/2608.11749v1) describes retaining MGDA/FAMO aggregation and replacing Adam with Muon. Our combinations use the released MOON parameter grouping; they are local implementations of those described baselines.
- **Eta ablation:** Appendix F.5, Table 8 uses beta values `1e-5, 5e-5, 1e-4, 5e-4, 1e-3`. We use those numeric values for **our entropic weight step eta**; eta is not the model LR and the two algorithms' weight updates are different.

The report uses final-epoch accuracy, with the released trainer's batch-average convention, consistently across methods. It also saves sample-weighted accuracy in each run. Gap is the final inner simplex Frank–Wolfe gap on a fixed 256-image **training** probe, not a left/right accuracy difference. Raw gap magnitudes depend on the oracle geometry.

**No new hyperparameter search is performed for baselines or ablation variants.** The saved validation-selected model LR, eta and momentum injection are held fixed for our ablations except for the one setting being ablated.


In [ ]:
#@title 7. Prepare all remaining runs and inspect completion status
from IPython.display import display
import pandas as pd
from experiments.multimnist.trainer import parser as training_parser, CONFIG as TRAIN_CONFIG

BASELINE_CONFIGS = {
    "moon": {"lr": 1e-3, "weight_lr": 1e-4, "gamma": 1e-3},
    "famo": {"lr": 1e-3, "weight_lr": 0.025, "gamma": 0.01},
    "famo_muon": {"lr": 1e-3, "weight_lr": 0.025, "gamma": 0.01},
    "mgda": {"lr": 1e-3},
    "mgda_muon": {"lr": 1e-3},
    "muon_ls": {"lr": 1e-3},
}
ETA_GRID = [1e-5, 5e-5, 1e-4, 5e-4, 1e-3]
assert ETA_GRID == TRAIN_CONFIG["eta_grid"], "Eta grid differs from the report's expected tags."
assert len(SEEDS) == len(set(SEEDS)) == 3
assert EPOCHS == 100 and BATCH_SIZE == 256, "Use the fixed reference protocol for this table."
DATA_SHA256 = data_fingerprint(DATA)

def cli_flags(settings):
    return [part for key, value in settings.items()
            for part in ("--" + key.replace("_", "-"), str(value))]

# Each entry describes one configuration; seeds are expanded below.
TABLE_JOBS = [{"group": "ours", "tag": "main", "method": "ours", "flags": list(OURS_FLAGS)}]
for method, settings in BASELINE_CONFIGS.items():
    group = "moon" if method == "moon" else "famo" if method.startswith("famo") else "other_baselines"
    TABLE_JOBS.append({"group": group, "tag": "main", "method": method, "flags": cli_flags(settings)})
for group, flag, values in [
    ("oracle", "--oracle", ["l2", "sign", "spectral"]),
    ("weights", "--weights", ["entropic", "projected"]),
    ("momentum", "--momentum", ["blended", "per-task", "none"]),
    ("eta", "--eta", ETA_GRID),
]:
    for value in values:
        TABLE_JOBS.append({"group": group, "tag": f"{group}-{value}", "method": "ours",
                           "flags": [*OURS_FLAGS, flag, str(value)]})
assert len(TABLE_JOBS) == 20

def command_args(job, seed):
    # The single changed ablation option comes last, overriding its base value.
    return ["--method", job["method"], "--tag", job["tag"], "--seed", seed,
            "--output-root", OUTPUT_ROOT, *COMMON, *job["flags"], "--resume"]

def expected_signature(job, seed):
    args = training_parser().parse_args(list(map(str, command_args(job, seed))))
    if args.lr is None:
        args.lr = TRAIN_CONFIG["ours"]["lr"] if args.method == "ours" else TRAIN_CONFIG["common"]["lr"]
    defaults = TRAIN_CONFIG["moon" if args.method == "moon" else "famo"]
    if args.weight_lr is None:
        args.weight_lr = defaults["weight_lr"]
    if args.gamma is None:
        args.gamma = defaults["gamma"]
    ignored = {"resume", "stop_after_epoch", "output_root", "data_path", "device", "workers", "threads", "cache_data"}
    signature = {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items() if k not in ignored}
    signature.update(data_sha256=DATA_SHA256, implementation="multimnist-v2",
                     source_commit=TRAIN_CONFIG["source_commit"])
    return signature

EXPECTED_SIGNATURES = {(job["tag"], job["method"], seed): expected_signature(job, seed)
                       for job in TABLE_JOBS for seed in SEEDS}

def run_status(job, seed):
    directory = OUTPUT_ROOT / job["tag"] / job["method"] / f"seed{seed}"
    summary_path = directory / "summary.json"
    expected = EXPECTED_SIGNATURES[job["tag"], job["method"], seed]
    # Validate even partial configuration files before scheduling more work.
    for filename in ("config.json", "summary.json"):
        path = directory / filename
        if path.exists():
            saved = json.loads(path.read_text())
            if saved.get("signature") != expected:
                raise ValueError(f"Configuration/data mismatch: {path}. Keep the original settings or use a separate experiment directory.")
    row = {"group": job["group"], "tag": job["tag"], "method": job["method"], "seed": seed,
           "status": "pending", "epoch": 0, "remaining_epochs": EPOCHS}
    if summary_path.exists():
        saved = json.loads(summary_path.read_text())
        assert saved["method"] == job["method"] and saved["tag"] == job["tag"] and saved["seed"] == seed
        assert saved.get("selection") == "test" and not saved.get("smoke"), f"Not a final test run: {summary_path}"
        epoch = int(saved["epoch"])
        assert 0 <= epoch <= EPOCHS
        complete = saved.get("completed") is True and epoch == EPOCHS
        row.update(status="complete" if complete else "resume", epoch=epoch, remaining_epochs=EPOCHS-epoch)
        if not complete and not (directory / "checkpoint.pt").exists():
            raise FileNotFoundError(f"Partial run has no checkpoint: {directory}")
    elif (directory / "checkpoint.pt").exists():
        row["status"] = "resume (checkpoint)"
    return row

def table_status(show=True):
    frame = pd.DataFrame([run_status(job, seed) for job in TABLE_JOBS for seed in SEEDS])
    frame.to_csv(OUTPUT_ROOT / "table_run_status.csv", index=False)
    if show:
        display(frame.groupby(["group", "status"], sort=False).agg(
            runs=("seed", "size"), remaining_epochs=("remaining_epochs", "sum")).reset_index())
        print(f"Complete: {(frame.status == 'complete').sum()}/{len(frame)} runs; "
              f"remaining epoch estimate: {int(frame.remaining_epochs.sum())}.", flush=True)
    return frame

def run_group(group):
    jobs = [job for job in TABLE_JOBS if job["group"] == group]
    assert jobs, f"Unknown group: {group}"
    started = time.monotonic()
    for index, (job, seed) in enumerate(((j, s) for j in jobs for s in SEEDS), 1):
        status = run_status(job, seed)
        label = f"{job['tag']}/{job['method']}/seed{seed}"
        print(f"\n[{group} {index}/{len(jobs)*len(SEEDS)}] {label}: {status['status']}", flush=True)
        if status["status"] == "complete":
            print("Skipping matching completed run.", flush=True)
            continue
        run_script("run_multimnist.py", *command_args(job, seed))
        assert run_status(job, seed)["status"] == "complete", f"Run did not finish: {label}"
    print(f"Group {group} finished in {(time.monotonic()-started)/60:.1f} minutes.", flush=True)
    table_status()

protocol = {
    "version": 1, "training_commit": REPO_COMMIT, "data_sha256": DATA_SHA256,
    "epochs": EPOCHS, "batch_size": BATCH_SIZE, "seeds": SEEDS,
    "ours_selected": SELECTED, "baseline_configs": BASELINE_CONFIGS, "eta_grid": ETA_GRID,
    "jobs": [{k: list(map(str, v)) if k == "flags" else v for k, v in job.items()} for job in TABLE_JOBS],
    "sources": {"paper": "https://arxiv.org/pdf/2608.11749v1",
                "moon_commit": TRAIN_CONFIG["source_commit"],
                "moon": "experiments/multimnist/run_muon_vit.sh",
                "famo": "experiments/utils.py: common_parser defaults",
                "muon_combinations": "Appendix F.2, Table 5; released MOON optimizer grouping",
                "eta_grid": "Appendix F.5, Table 8 (MOON beta values applied to Ours eta)"},
}
protocol_path = RUN_ROOT / "full_table_protocol.json"
if protocol_path.exists():
    assert json.loads(protocol_path.read_text()) == protocol, "Table protocol changed; keep settings fixed."
else:
    atomic_json(protocol_path, protocol)
status = table_status()
print("Ready. Run cells 8–14 for all remaining groups, then cell 15 for the final tables.")


In [ ]:
#@title 8. MOON — three seeds, official launch settings
run_group("moon")


In [ ]:
#@title 9. FAMO and FAMO + Muon — three seeds each
run_group("famo")


In [ ]:
#@title 10. MGDA, MGDA + Muon and Muon (LS) — three seeds each
run_group("other_baselines")


In [ ]:
#@title 11. Oracle ablation — l2, sign and spectral
run_group("oracle")


In [ ]:
#@title 12. Weight-update ablation — entropic and projected
run_group("weights")


In [ ]:
#@title 13. Momentum ablation — blended, per-task and none
run_group("momentum")


In [ ]:
#@title 14. Weight-step ablation — five values from the MOON beta grid
run_group("eta")


In [ ]:
#@title 15. Export all measured results, complete LaTeX tables and run status
# Safe to rerun after any group; incomplete configurations remain --.
status = table_status()
REPORT = OUTPUT_ROOT / "report"
LOCAL_REPORT = OUTPUT_ROOT / "report_local_only"
for baseline_source, output in [("reproduced", LOCAL_REPORT), ("reported", REPORT)]:
    run_script("report_multimnist.py", "--root", OUTPUT_ROOT, "--seeds", *SEEDS,
               "--baseline-source", baseline_source, "--out", output)

results = pd.read_csv(LOCAL_REPORT / "results.csv")
print("Comparison: locally measured final test results (not copied paper values)")
display(results.loc[results.tag == "main", ["method", "n_seeds", "left", "right", "avg", "avg_std"]])
print("Ablations: each variant has its own mean accuracy and final inner gap")
display(results.loc[results.tag != "main", ["tag", "n_seeds", "avg", "avg_std", "gap", "gap_std"]])
missing = status.loc[status.status != "complete"]
if missing.empty:
    assert len(results) == 20 and (results.n_seeds == 3).all()
    assert "--" not in (LOCAL_REPORT / "table.tex").read_text().split(r"\caption{")[0]
    print("COMPLETE: all 7 comparison rows and all 13 ablation variants have three seeds.")
else:
    print(f"PARTIAL: {len(missing)} runs remain. Run the corresponding group cells, then rerun cell 15.")
    display(missing[["group", "tag", "method", "seed", "status", "epoch"]])

# A small downloadable bundle with tables, provenance, and per-seed JSON metrics.
# Full model checkpoints stay in Drive and are not duplicated into this ZIP.
BUNDLE = RUN_ROOT / "multimnist_full_table_reports.zip"
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as archive:
    paths = [RUN_ROOT / "experiment.json", RUN_ROOT / "final_configuration.json",
             RUN_ROOT / "full_table_protocol.json", SELECTION_FILE, OUTPUT_ROOT / "table_run_status.csv"]
    paths += [path for folder in (REPORT, LOCAL_REPORT) for path in sorted(folder.iterdir()) if path.is_file()]
    for job in TABLE_JOBS:
        for seed in SEEDS:
            folder = OUTPUT_ROOT / job["tag"] / job["method"] / f"seed{seed}"
            paths += [folder / name for name in ("config.json", "summary.json")]
    for path in paths:
        if path.exists():
            archive.write(path, path.relative_to(RUN_ROOT))

print("Use this LaTeX for the fully measured comparison:", LOCAL_REPORT / "table.tex")
print("Alternative with published baseline numbers:", REPORT / "table.tex")
print("Measured CSV:", LOCAL_REPORT / "results.csv")
print("Report bundle in Drive:", BUNDLE)


## Resume and retrieve results

- **Existing Ours results:** leave `EXPERIMENT_NAME = "multimnist_a100_v2"`, `DATA_CACHE_EXPERIMENT = "multimnist_a100_v1"`, and `RUN_TUNING = False`. Run **1–5 → 7 → 8–14 → 15**. Cell 6 is optional when Ours has already finished.
- **After a disconnection:** run **1–5 and 7** again, then the interrupted group cell and any remaining groups. Completed runs are skipped after their configuration and dataset signatures are checked. An unfinished epoch may repeat.
- **Main comparison only:** run **8–10**, then **15**. The ablation cells can be run later.
- **Ablations only:** run **11–14**, then **15**. Each group uses the same frozen Ours configuration and changes one setting.
- **Run all:** with saved tuning/Ours results, this also works; cell 6 checks the existing final checkpoints without training more epochs. Running all remaining groups can take several hours. Cell 7 shows the actual remaining budget.
- **Files in Drive:** `MyDrive/LMO-MOO/multimnist_a100_v2/`. The `multimnist_full_table_reports.zip` file contains both LaTeX versions, measured CSVs, run status, exact settings and per-seed summaries. Download it from Drive when ready.
- **Table choice:** `results/multimnist/report_local_only/table.tex` contains local measurements for all methods. `results/multimnist/report/table.tex` keeps the paper's published baseline numbers and clearly labels their origin. Missing three-seed configurations remain `--`; no scores are inferred.
- **Logs:** each subprocess streams its output live and saves a separate log in `logs/`. Training code prints only end-of-epoch metrics.
- **Finish:** release the GPU with **Runtime → Disconnect and delete runtime**. Drive outputs remain available.
